# Importing Libraries & Installing Dep.

In [ ]:
!pip install flask joblib
!pip install pyngrok
!ngrok authtoken 2rF3GEnzxoREZw1N2MSQji59AHA_4AUozkWuRyHibQwzGn7Bi
!pip install gunicorn
!pip install fuzzywuzzy
!pip install gunicorn
!pip install waitress
!pip install tensorflow_decision_forests
!pip install pandas scikit-learn fuzzywuzzy Flask joblib
!pip install flask-cors

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 4.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of tf-keras to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.3/615.3 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 93.5 MB/s eta 0:00:00
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.17.1
    Uninstalling tensorboard-2.17.1:
      Successfully uninstalled tensorboard-2.17.1
  Attempting uninstall: tensorflow
    Found existing installatio

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from fuzzywuzzy import process
from flask import Flask, request, jsonify
import joblib
import numpy as np
from pyngrok import ngrok
import threading
import time

# Load Dataset

In [ ]:
df = pd.read_excel("appointments_dataset.xlsx")

# Data Pre-Processing

Getting to know the data

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
 df.describe()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df['Appointment Type'] = df['Appointment Type'].str.lower().str.strip()

In [ ]:
# Encoding 'Appointment Type' column to numeric format using LabelEncoder
label_encoder = LabelEncoder()
df['Appointment Type Encoded'] = label_encoder.fit_transform(df['Appointment Type'])

In [ ]:
# Features (Appointment Type Encoded) and Target (Priority Score)
X = df[['Appointment Type Encoded']]

y = df['Priority']

In [ ]:
X

In [ ]:
y

# Splitting data into Training and Testing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train

In [ ]:
y_test

# Training Model - RandomForestRegressor

Why this model?

*this is a regression problem where RandomForestRegressor is a suitable choice. The model learns the patterns between different appointment types and their corresponding priority scores, allowing it to predict the priority for new and unseen appointment types.*

In [ ]:
# Create and train the model (RandomForestRegressor)
model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

In [ ]:
# Predict on the test set to check performance
y_pred = model.predict(X_test)

# Evaluating the model

In [ ]:
# Calculate the mean squared error for regression models
mse = mean_squared_error(y_test, y_pred)
print(f"Model Mean Squared Error: {mse:.2f}")

Training Accuracy

In [32]:
# Calculate the R-squared score for the training data
y_train_pred = model.predict(X_train)
from sklearn.metrics import r2_score
r2_train = r2_score(y_train, y_train_pred)

print(f"Model Training R-squared Score: {r2_train:.2f}")

Model Training R-squared Score: 0.88


Testing Accuracy

In [33]:
# Calculate the R-squared score for the test data (testing accuracy)
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)

print(f"Model Testing R-squared Score: {r2:.2f}")

Model Testing R-squared Score: 0.84


# Saving the model

In [ ]:
# Save the model and the label encoder
joblib.dump(model, "priority_model.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")
joblib.dump(label_encoder, 'appointment_label_encoder.pkl')

# Communicating From APP to Model vise versa

In [ ]:
# Load the saved model and label encoder
model = joblib.load("priority_model.pkl")
label_encoder = joblib.load("label_encoder.pkl")

In [ ]:
df['Appointment Type'] = df['Appointment Type'].str.lower().str.strip()

Communication route using flask hosting in ngrok

In [ ]:
'''from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)  # Enables CORS for all routes

@app.route('/predict_priority_v2', methods=['POST'])
def predict_priority_v2():
    try:
        # Get JSON payload
        data = request.get_json()
        if not data:
            return jsonify({'error': 'No JSON data provided'}), 400

        print(f"Received data: {data}")
        appointment_type = data.get('appointment_type', '').lower().strip()

        if not appointment_type:
            return jsonify({'error': 'Appointment type is missing'}), 400

        # Mock prediction logic
        predicted_priority = 0.8  #
        return jsonify({'predicted_priority': predicted_priority})

    except Exception as e:
        print(f"Error: {e}")
        return jsonify({'error': str(e)}), 500

if __name__ == '__main__':
    app.run(debug=True)'''



starting ngrok in a separate thread so Keep the script running or ngrok will terminate

In [ ]:
'''def start_ngrok():
    ngrok_tunnel = ngrok.connect(5000, bind_tls=True)
    print(f" * Running on {ngrok_tunnel.public_url}")
    while True:
        time.sleep(1)


ngrok_thread = threading.Thread(target=start_ngrok, daemon=True)
ngrok_thread.start()
if __name__ == '__main__':
       app.run(debug=True, use_reloader=False)'''

# Example for for predicting new appointment

In [ ]:
from fuzzywuzzy import process

def predict_priority_for_new_appointment(new_appointment):
    new_appointment = new_appointment.lower().strip()

    # Fuzzy matching to find the closest match
    closest_match = process.extractOne(new_appointment, df['Appointment Type'])

    if closest_match:
        matched_appointment = closest_match[0]
        print(f"Matched Appointment Type: {matched_appointment}")

        new_appointment_encoded = label_encoder.transform([matched_appointment])

        # Ensure the input data has the same structure as the training data
        new_data = pd.DataFrame({'Appointment Type Encoded': new_appointment_encoded})

        # Predict priority
        predicted_priority = model.predict(new_data)
        return predicted_priority[0]
    else:
        return None

In [ ]:
new_appointment = input("Enter the appointment type: ")

predicted_priority = predict_priority_for_new_appointment(new_appointment)

if predicted_priority is not None:
    print(f"Predicted Priority Score for '{new_appointment}': {predicted_priority}")